# PyTorch GPU / CPU Benchmark

Based on [tommykho/pytorch-benchmark](https://github.com/tommykho/pytorch-benchmark).

Benchmarks PyTorch performance for both training and inference across CPU and CUDA devices.
Tests matrix operations, convolutions, and a simple MLP — giving you a quick read on relative throughput.

**Requirements:** `pip install torch torchvision psutil tabulate py-cpuinfo`

In [ ]:
# !pip install torch torchvision psutil tabulate py-cpuinfo

In [ ]:
import torch
import time
import psutil
from tabulate import tabulate

try:
    import cpuinfo
    cpu_name = cpuinfo.get_cpu_info().get("brand_raw", "Unknown CPU")
except ImportError:
    cpu_name = "Unknown CPU"

print(f"PyTorch version : {torch.__version__}")
print(f"CPU             : {cpu_name}")
print(f"RAM             : {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM            : {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")

## Benchmark helpers

In [ ]:
def bench(fn, device, warmup=3, runs=10):
    """Run fn() on device, return mean wall-clock time in ms."""
    # warmup
    for _ in range(warmup):
        fn()
        if device == "cuda":
            torch.cuda.synchronize()

    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    mean_ms = sum(times) / len(times)
    return mean_ms


def run_benchmark(devices=None):
    if devices is None:
        devices = ["cpu"]
        if torch.cuda.is_available():
            devices.append("cuda")

    results = []

    for device in devices:
        # ---- Matrix multiply (4096 x 4096) ----
        N = 4096
        a = torch.randn(N, N, device=device)
        b = torch.randn(N, N, device=device)
        t_matmul = bench(lambda: torch.mm(a, b), device)

        # ---- Conv2d (batch=32, 64 channels, 224x224) ----
        import torch.nn as nn
        conv = nn.Conv2d(64, 64, kernel_size=3, padding=1).to(device)
        x_conv = torch.randn(32, 64, 224, 224, device=device)
        t_conv = bench(lambda: conv(x_conv), device)

        # ---- MLP forward (4-layer, 2048 hidden) ----
        mlp = nn.Sequential(
            nn.Linear(2048, 2048), nn.ReLU(),
            nn.Linear(2048, 2048), nn.ReLU(),
            nn.Linear(2048, 2048), nn.ReLU(),
            nn.Linear(2048, 1000),
        ).to(device)
        x_mlp = torch.randn(512, 2048, device=device)
        t_mlp = bench(lambda: mlp(x_mlp), device)

        # ---- MLP training step ----
        optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()
        labels = torch.randint(0, 1000, (512,), device=device)

        def train_step():
            optimizer.zero_grad()
            out = mlp(x_mlp)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

        t_train = bench(train_step, device)

        results.append({
            "Device": device.upper(),
            "MatMul 4096² (ms)": f"{t_matmul:.1f}",
            "Conv2d 224² (ms)": f"{t_conv:.1f}",
            "MLP fwd (ms)": f"{t_mlp:.1f}",
            "MLP train (ms)": f"{t_train:.1f}",
        })

    print(tabulate(results, headers="keys", tablefmt="github"))
    return results


results = run_benchmark()

## Throughput summary (samples / second)

In [ ]:
# Speedup ratio CPU → GPU
if torch.cuda.is_available() and len(results) == 2:
    cpu_r, gpu_r = results[0], results[1]
    headers = ["Benchmark", "CPU (ms)", "GPU (ms)", "Speedup"]
    rows = []
    for key in ["MatMul 4096² (ms)", "Conv2d 224² (ms)", "MLP fwd (ms)", "MLP train (ms)"]:
        cpu_ms = float(cpu_r[key])
        gpu_ms = float(gpu_r[key])
        speedup = cpu_ms / gpu_ms
        rows.append([key, f"{cpu_ms:.1f}", f"{gpu_ms:.1f}", f"{speedup:.1f}x"])
    print(tabulate(rows, headers=headers, tablefmt="github"))
else:
    print("Only one device available — no speedup comparison.")

## Memory usage

In [ ]:
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated() / 1e6
    reserved = torch.cuda.memory_reserved() / 1e6
    print(f"GPU memory allocated : {alloc:.1f} MB")
    print(f"GPU memory reserved  : {reserved:.1f} MB")

ram = psutil.virtual_memory()
print(f"RAM used             : {ram.used / 1e9:.1f} / {ram.total / 1e9:.1f} GB  ({ram.percent:.1f}%)")